# PDF Scraper

## Housekeeping

In [11]:
# Libraries
import fitz
from transformers import (TrOCRConfig,
    TrOCRProcessor,
    TrOCRForCausalLM,
    ViTConfig,
    ViTModel,
    VisionEncoderDecoderModel)
from huggingface_hub import login
from PIL import Image
import os
import pandas as pd
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import re


In [27]:
# Paths

test_dataset = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/PDF_Scraping/" #test_dataset/test_dataset/"

os.chdir("C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/")
states_ref = pd.read_csv("states_ref.csv")

login()

## Extracting Information

In [28]:
# info = defaultdict(list)
# for dirpath, dirs, files in os.walk(test_dataset):
#     for file in files:
#         file_name = os.path.join(dirpath, file).split("/")[-1]
#         print(file_name)
#         if ".pdf" in file_name:
#             info["animal"].append(file_name.split(" ")[1])
#             info["sending_state"].append(file_name.split(" ")[2])
#             info["receiving_state"].append(file_name.split(" ")[3])
#             info["date"].append(dateutil.parser.parse(file_name.split(" ")[4][0:2] + "-" + file_name.split(" ")[4][2:4] + "-" + file_name.split(" ")[4][4:] ,dayfirst=False))
            
#     break 

# info_df = pd.DataFrame.from_dict(info)
# print(info_df)
# info_df["year"] = info_df["date"].apply(lambda x: x.year)

# print(info_df)

os.chdir(test_dataset)
# info_df.to_csv("pdf_info.csv")

In [32]:
encoder = ViTModel(ViTConfig())
decoder = TrOCRForCausalLM(TrOCRConfig())
model = VisionEncoderDecoderModel(encoder=encoder, decoder=decoder)

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten')

pdf_name = "00062_41-1712766 41-1712767 41-1712770 BOV MN IL 05142021 View"
spreadsheet = "MN_CVI_example_spreadsheet.csv"

doc = fitz.open(pdf_name + ".pdf") # open document

file_name = ""
for i, page in enumerate(doc):
    # Render page to a pixmap (image representation)
    pix = page.get_pixmap()
    # Save the pixmap as an image file
    file_name = pdf_name + f"_page_{i}.png"
    pix.save(file_name)

doc.close()

print(file_name)
image = Image.open(file_name).convert("RGB")
pixel_values = processor(images=image, return_tensors="pt").pixel_values

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


00062_41-1712766 41-1712767 41-1712770 BOV MN IL 05142021 View_page_2.png


In [ ]:
spreadsheet_df = pd.read_csv(spreadsheet)
origin_city = spreadsheet_df["origin_city"].values[0]
print(origin_city)
# text = "industry, ' Mr. Brown commented icily. ' Let us have a"

# training
model.config.decoder_start_token_id = processor.tokenizer.eos_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

labels = processor.tokenizer(origin_city, return_tensors="pt").input_ids
outputs = model(pixel_values, labels=labels)
loss = outputs.loss
round(loss.item(), 2)

Bovey


TypeError: PreTokenizedEncodeInput must be Union[PreTokenizedInputSequence, Tuple[PreTokenizedInputSequence, PreTokenizedInputSequence]]

In [31]:
generated_ids = model.generate(pixel_values)
text = processor.batch_decode(generated_ids, skip_special_tokens=True) # [0]

print(text)

C:\Users\maksiaevai.NCBI_NT\AppData\Roaming\Python\Python313\site-packages\transformers\generation\utils.py:1551: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


['# the amount of the']
